# P02 — Auditoría y mitigación de fairness con Fairlearn

En P01 evaluamos el rendimiento global del modelo de Velora Finance. Ahora auditamos si ese comportamiento es comparable entre grupos y analizamos qué ocurre cuando intentamos **mitigar** una disparidad.

`customer_group` se utiliza para auditoría y postprocesado; no forma parte de las variables predictoras del modelo base.

La práctica utiliza **Fairlearn 0.15.0**. El código está preparado: la interacción se centra en cambiar decisiones y comparar resultados.


## 1. Preparar Fairlearn y recrear el caso Velora


In [ ]:
!pip -q install fairlearn==0.15.0


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    demographic_parity_difference,
    demographic_parity_ratio,
    equal_opportunity_difference,
    equalized_odds_difference,
    false_positive_rate,
    false_negative_rate,
)
from fairlearn.postprocessing import ThresholdOptimizer

pd.set_option("display.max_columns", None)

GENERATOR_URL = "https://raw.githubusercontent.com/pacosepulveda/ia-confiable-explicable-labs/main/resources/generate_velora_credit.py"
namespace = {"__name__": "velora_generator"}
with urllib.request.urlopen(GENERATOR_URL) as response:
    exec(response.read().decode("utf-8"), namespace)

df = namespace["generate_dataset"]()
print("Dataset:", df.shape)
print("Fairlearn preparado.")


## 2. Recrear exactamente el modelo de P01

Usamos el mismo dataset, variables y partición train/test de P01. Así la primera auditoría parte del mismo sistema que ya conocemos.


In [ ]:
FEATURES = [
    "age", "annual_income", "debt_ratio", "employment_months",
    "late_payments_12m", "credit_history_months", "existing_debt",
    "requested_amount", "savings_balance",
]

X = df[FEATURES]
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

audit = df.loc[X_test.index, ["application_id", "customer_group", "default"]].copy()
audit["prob_default"] = proba

print("Modelo de P01 recreado.")
print("customer_group es predictor:", "customer_group" in FEATURES)


## 3. Contexto antes de medir fairness

Antes de comparar predicciones observamos cuántos casos hay en cada grupo, la tasa real de impago y el riesgo medio predicho.

Las **tasas base** importan: distintos criterios de fairness pueden ser incompatibles o contar historias distintas cuando los grupos parten de distribuciones diferentes.


In [ ]:
context = audit.groupby("customer_group").agg(
    n=("default", "size"),
    tasa_real_impago=("default", "mean"),
    riesgo_medio_predicho=("prob_default", "mean"),
)

display(context.style.format({
    "tasa_real_impago": "{:.2%}",
    "riesgo_medio_predicho": "{:.2%}",
}))


## 4. Auditoría con `MetricFrame`

Para esta auditoría definimos el resultado favorable como **solvente/aprobado = 1**:

- `solvente_real = 1 - default`
- `aprobado = 1 - pred_default`

Esto evita interpretar automáticamente la clase positiva técnica como el resultado favorable del proceso.


In [ ]:
def audit_threshold(threshold):
    out = audit.copy()
    out["pred_default"] = (out["prob_default"] >= threshold).astype(int)
    out["solvente_real"] = 1 - out["default"]
    out["aprobado"] = 1 - out["pred_default"]

    metrics = {
        "accuracy": accuracy_score,
        "tasa_aprobacion": selection_rate,
        "recall_solventes": recall_score,
        "precision_aprobacion": precision_score,
        "FPR_aprobacion": false_positive_rate,
        "FNR_aprobacion": false_negative_rate,
    }

    frame = MetricFrame(
        metrics=metrics,
        y_true=out["solvente_real"],
        y_pred=out["aprobado"],
        sensitive_features=out["customer_group"],
    )

    fairness = pd.Series({
        "demographic_parity_difference":
            demographic_parity_difference(
                out["solvente_real"], out["aprobado"],
                sensitive_features=out["customer_group"]
            ),
        "demographic_parity_ratio":
            demographic_parity_ratio(
                out["solvente_real"], out["aprobado"],
                sensitive_features=out["customer_group"]
            ),
        "equal_opportunity_difference":
            equal_opportunity_difference(
                out["solvente_real"], out["aprobado"],
                sensitive_features=out["customer_group"]
            ),
        "equalized_odds_difference":
            equalized_odds_difference(
                out["solvente_real"], out["aprobado"],
                sensitive_features=out["customer_group"]
            ),
    })

    return out, frame, fairness

audit_050, frame_050, fairness_050 = audit_threshold(0.50)

print("Métricas globales")
display(frame_050.overall.to_frame("valor").style.format("{:.2%}"))

print("Métricas por grupo")
display(frame_050.by_group.style.format("{:.2%}"))

print("Diferencias/ratios de fairness")
display(fairness_050.to_frame("valor").style.format("{:.3f}"))


### Interpretación

No buscamos que todos los valores sean idénticos.

Una tasa de aprobación relativamente parecida puede coexistir con diferencias relevantes en los errores. Por eso **fairness no debe reducirse a una única cifra**.


## 5. Visualizar la distribución de errores


In [ ]:
plot_table = frame_050.by_group[
    ["tasa_aprobacion", "FPR_aprobacion", "FNR_aprobacion"]
]

plot_table.plot(kind="bar", figsize=(8, 5))
plt.ylabel("Tasa")
plt.title("Métricas por grupo — threshold 0.50")
plt.xticks(rotation=0)
plt.show()


## 6. Cambiar el threshold

El modelo no cambia. Cambia la política que transforma el score en una decisión.

Observa cómo una decisión operacional puede modificar simultáneamente rendimiento y fairness.


In [ ]:
THRESHOLD_ELEGIDO = 0.50 #@param {type:"slider", min:0.10, max:0.80, step:0.05}

_, selected_frame, selected_fairness = audit_threshold(THRESHOLD_ELEGIDO)

print(f"Threshold: {THRESHOLD_ELEGIDO:.2f}")
print()
print("Por grupo")
display(selected_frame.by_group.style.format("{:.2%}"))

print()
print("Fairness")
display(selected_fairness.to_frame("valor").style.format("{:.3f}"))


### Preguntas

- ¿Hay algún threshold global que haga desaparecer todas las diferencias?
- ¿Mejorar demographic parity mejora necesariamente equalized odds?
- ¿Qué error importa más en este caso?
- ¿Qué criterio podría justificarse ante riesgo, negocio y auditoría?

**No existe un criterio de fairness universalmente correcto.**


## 7. Calibración por grupo

Fairness de tasas y errores no es lo mismo que **calibración**.

Si el modelo asigna, por ejemplo, un riesgo cercano al 30 %, queremos comprobar si la frecuencia observada de impago en ese rango es aproximadamente coherente.

Agrupamos las probabilidades en intervalos comunes para comparar grupos.


In [ ]:
calibration = audit.copy()

edges = np.unique(
    np.quantile(calibration["prob_default"], [0, .2, .4, .6, .8, 1.0])
)

calibration["bin"] = pd.cut(
    calibration["prob_default"],
    bins=edges,
    include_lowest=True,
    duplicates="drop",
)

calibration_table = (
    calibration
    .groupby(["customer_group", "bin"], observed=True)
    .agg(
        n=("default", "size"),
        riesgo_predicho=("prob_default", "mean"),
        impago_observado=("default", "mean"),
    )
    .reset_index()
)

display(calibration_table.style.format({
    "riesgo_predicho": "{:.2%}",
    "impago_observado": "{:.2%}",
}))


In [ ]:
for group in sorted(calibration_table["customer_group"].unique()):
    part = calibration_table[calibration_table["customer_group"] == group]
    plt.plot(
        part["riesgo_predicho"],
        part["impago_observado"],
        marker="o",
        label=f"Grupo {group}",
    )

max_value = max(
    calibration_table["riesgo_predicho"].max(),
    calibration_table["impago_observado"].max(),
)
plt.plot(
    [0, max_value], [0, max_value],
    "--",
    label="Calibración ideal",
)
plt.xlabel("Riesgo medio predicho")
plt.ylabel("Impago observado")
plt.title("Calibración por grupo")
plt.legend()
plt.show()


### Interpretación

Un sistema puede estar razonablemente calibrado y no satisfacer demographic parity o equalized odds.

La calibración añade otra pregunta:

> Cuando el modelo dice “riesgo X %”, ¿ese número conserva un significado comparable?

De nuevo, **una dimensión no sustituye a las demás**.


# 8. De auditoría a mitigación

Detectar una disparidad no responde todavía a la pregunta:

> ¿Qué hacemos ahora?

Vamos a probar una mitigación real de Fairlearn mediante `ThresholdOptimizer`.

Para hacerlo de forma más realista necesitamos **tres conjuntos**:

1. entrenamiento del modelo;
2. calibración del postprocesado;
3. test final.

No utilizamos el test final para elegir los umbrales.


In [ ]:
X_dev, X_final, y_dev, y_final = train_test_split(
    df[FEATURES],
    df["default"],
    test_size=0.20,
    random_state=42,
    stratify=df["default"],
)

X_fit, X_mit, y_fit, y_mit = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,      # 20% del total
    random_state=42,
    stratify=y_dev,
)

base_mit_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

base_mit_model.fit(X_fit, y_fit)

groups_mit = df.loc[X_mit.index, "customer_group"]
groups_final = df.loc[X_final.index, "customer_group"]

print("Train modelo :", len(X_fit))
print("Calibración  :", len(X_mit))
print("Test final   :", len(X_final))


## 9. `ThresholdOptimizer` con restricción de equalized odds

`ThresholdOptimizer` no reentrena el modelo base. Aprende una política de postprocesado que utiliza el score del estimador y el grupo para satisfacer una restricción de fairness.

Aquí utilizamos:

- `constraints="equalized_odds"`
- `objective="accuracy_score"`

Esto es una **intervención concreta**, no una garantía de que el sistema resultante sea globalmente “justo”.


In [ ]:
postprocessor = ThresholdOptimizer(
    estimator=base_mit_model,
    constraints="equalized_odds",
    objective="accuracy_score",
    prefit=True,
    predict_method="predict_proba",
)

postprocessor.fit(
    X_mit,
    y_mit,
    sensitive_features=groups_mit,
)

print("Postprocesador ajustado.")


## 10. Comparar antes y después sobre test final

Evaluamos el modelo base y el sistema postprocesado **sobre el mismo test final**.


In [ ]:
base_default = base_mit_model.predict(X_final)

mitigated_default = postprocessor.predict(
    X_final,
    sensitive_features=groups_final,
    random_state=42,
)

def evaluate_default_predictions(y_true, y_pred, groups):
    # Para Fairlearn convertimos a resultado favorable: solvente/aprobado = 1
    solvent_true = 1 - np.asarray(y_true)
    approved = 1 - np.asarray(y_pred)

    frame = MetricFrame(
        metrics={
            "accuracy": accuracy_score,
            "tasa_aprobacion": selection_rate,
            "recall_solventes": recall_score,
            "FPR_aprobacion": false_positive_rate,
            "FNR_aprobacion": false_negative_rate,
        },
        y_true=solvent_true,
        y_pred=approved,
        sensitive_features=np.asarray(groups),
    )

    fairness = pd.Series({
        "demographic_parity_difference":
            demographic_parity_difference(
                solvent_true, approved,
                sensitive_features=np.asarray(groups)
            ),
        "equal_opportunity_difference":
            equal_opportunity_difference(
                solvent_true, approved,
                sensitive_features=np.asarray(groups)
            ),
        "equalized_odds_difference":
            equalized_odds_difference(
                solvent_true, approved,
                sensitive_features=np.asarray(groups)
            ),
    })

    return frame, fairness

base_frame, base_fair = evaluate_default_predictions(
    y_final, base_default, groups_final
)

mit_frame, mit_fair = evaluate_default_predictions(
    y_final, mitigated_default, groups_final
)

comparison_overall = pd.DataFrame({
    "Base": base_frame.overall,
    "Mitigado": mit_frame.overall,
})

comparison_fairness = pd.DataFrame({
    "Base": base_fair,
    "Mitigado": mit_fair,
})

print("Métricas globales")
display(comparison_overall.style.format("{:.2%}"))

print("Fairness")
display(comparison_fairness.style.format("{:.3f}"))

print("Por grupo — modelo base")
display(base_frame.by_group.style.format("{:.2%}"))

print("Por grupo — mitigado")
display(mit_frame.by_group.style.format("{:.2%}"))


## 11. ¿Hemos “arreglado” el sistema?

No necesariamente.

La mitigación puede reducir una disparidad bajo una definición concreta y, al mismo tiempo:

- cambiar accuracy;
- cambiar tasas de aprobación;
- redistribuir FP y FN;
- requerir conocer el grupo en tiempo de decisión;
- introducir una política más compleja;
- necesitar nuevas evidencias de gobernanza.

`ThresholdOptimizer` utiliza **umbrales específicos por grupo**. Esto convierte el atributo sensible en parte de la política de decisión, aunque no forme parte del modelo base.


In [ ]:
print("¿customer_group está en FEATURES? ->", "customer_group" in FEATURES)
print("¿ThresholdOptimizer necesita sensitive_features al predecir? -> Sí")


### Decisión de auditoría

Compara el sistema **Base** y el **Mitigado** y responde:

1. ¿Qué disparidad se ha reducido?
2. ¿Qué coste de rendimiento aparece?
3. ¿Qué errores se redistribuyen?
4. ¿Aceptarías una política con thresholds distintos por grupo?
5. ¿Qué documentación y aprobación requeriría esa decisión?

No basta con afirmar “la métrica mejoró”.


# Conclusiones de P02

- Las métricas globales pueden ocultar diferencias importantes.
- Eliminar una variable sensible del entrenamiento no demuestra fairness.
- Demographic parity, equal opportunity, equalized odds y calibración responden a preguntas diferentes.
- Fairness depende también de la política de decisión y del threshold.
- Una técnica de mitigación puede reducir una disparidad y crear nuevos trade-offs.
- Utilizar atributos sensibles para mitigar fairness puede ser técnicamente útil y, a la vez, requerir una justificación de gobierno especialmente cuidadosa.
- Una disparidad estadística es una señal para investigar; no es por sí sola una conclusión causal, ética o jurídica.
